# Local pressure-drop examples

This executed notebook is a compact, cell-by-cell reference for the public KalKalori local pressure-drop API. It covers reusable flow sections, distributed straight-section friction, local transitions, elbows, a planar obstruction, user-defined losses, ordered assembly, and a complete explicit tube-side path.

All inputs use SI units. Local paths are evaluated explicitly by the application layer; this notebook does not route them through `solve()`, `simulate()`, or `rate()`.

In [ ]:
import math
import sys
from pathlib import Path

import pandas as pd

repository_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "core").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from core.geometry import (
    AreaChangeGeometry,
    AreaChangeType,
    CircularFlowSection,
    CustomFlowSection,
    PressureDropAssemblyGeometry,
    RectangularFlowSection,
    SpecifiedTubeSidePressureDropPath,
    StraightSectionGeometry,
    UserDefinedPressureDropGeometry,
)
from core.pressure_drop import (
    CircularElbowGeometry,
    DirectionChangeMethod,
    ElbowConstruction,
    FlatObstructionGeometry,
    FlatObstructionType,
    PressureDropFlowState,
    RectangularElbowGeometry,
    RectangularTurnPlane,
    calculate_area_change_pressure_drop,
    calculate_circular_elbow_pressure_drop,
    calculate_flat_obstruction_pressure_drop,
    calculate_pressure_drop_assembly,
    calculate_rectangular_elbow_pressure_drop,
    calculate_straight_section_pressure_drop,
    calculate_tube_bundle_hydraulics,
    calculate_tube_side_pressure_drop_path,
    calculate_user_defined_pressure_drop,
)
from core.properties import GasMixturePropertyProvider, GasMixtureSpec

pd.options.display.float_format = "{:,.4f}".format
pd.options.display.max_colwidth = None

## Pressure-loss convention

For dynamic pressure `q = rho * V² / 2`, the local-path result uses:

- `delta_dynamic_pressure = q_out - q_in`
- `dp_static = dp_irreversible + delta_dynamic_pressure`

`dp_irreversible` is the hydraulic loss and is the only stage quantity summed as hydraulic resistance. `delta_dynamic_pressure` is not an additional loss. A negative `delta_dynamic_pressure` means that the flow decelerates. A negative `dp_static` means static pressure is recovered. Neither signed value implies a negative irreversible loss.

| Quantity | Meaning | Sign |
| --- | --- | --- |
| `dp_irreversible` | irreversible total-pressure loss | non-negative |
| `delta_dynamic_pressure` | `q_out - q_in` | signed |
| `dp_static` | `p_in - p_out` | signed |

In [ ]:
def show_stage_result(result):
    """Format diagnostics already present on a production stage result."""
    fields = (
        ("stage ID", result.stage_id),
        ("stage type", result.stage_type),
        ("status", result.status.value),
        ("method", result.method),
        ("reference area [m²]", result.reference_area),
        ("reference velocity [m/s]", result.reference_velocity),
        ("reference dynamic pressure [Pa]", result.reference_dynamic_pressure),
        ("loss coefficient K [-]", result.loss_coefficient),
        ("upstream area [m²]", result.upstream_area),
        ("downstream area [m²]", result.downstream_area),
        ("upstream velocity [m/s]", result.upstream_velocity),
        ("downstream velocity [m/s]", result.downstream_velocity),
        ("Reynolds number [-]", result.reynolds),
        ("relative roughness [-]", result.relative_roughness),
        ("Darcy friction factor [-]", result.friction_factor),
        ("friction-factor method", result.friction_factor_method),
        ("blockage ratio [-]", result.blockage_ratio),
        ("open-area ratio [-]", result.open_area_ratio),
        ("open-area velocity [m/s]", result.open_area_velocity),
        ("irreversible pressure loss [Pa]", result.dp_irreversible),
        ("dynamic-pressure change [Pa]", result.delta_dynamic_pressure),
        ("static-pressure difference [Pa]", result.dp_static),
        (
            "warnings",
            "; ".join(f"{warning.code}: {warning.message}" for warning in result.warnings)
            or "none",
        ),
    )
    return pd.DataFrame(
        [{"quantity": label, "value": value} for label, value in fields if value is not None]
    )


def assert_stage_consistent(result):
    """Check the public pressure convention without recreating a model equation."""
    assert result.dp_irreversible >= 0.0
    assert math.isclose(
        result.dp_static,
        result.dp_irreversible + result.delta_dynamic_pressure,
        rel_tol=1e-12,
        abs_tol=1e-12,
    )
    return result

## Fluid state

Local pressure-drop functions receive an already evaluated `PressureDropFlowState`; they do not perform a hidden property lookup. Here a production `GasMixturePropertyProvider` evaluates dry-air transport properties once at 30 °C and 101325 Pa absolute, and the resulting state is reused.

In [ ]:
air_provider = GasMixturePropertyProvider(
    GasMixtureSpec(
        components={"N2": 0.79, "O2": 0.21},
        basis="mole",
        backend="HEOS",
    )
)
air_state = PressureDropFlowState(
    mass_flow=5.0,  # kg/s
    temperature=303.15,  # 30 °C [K]
    pressure=101_325.0,  # absolute pressure [Pa]
    props=air_provider.at(T=303.15, p=101_325.0),
)

pd.DataFrame(
    {
        "quantity": [
            "mass flow",
            "temperature",
            "absolute pressure",
            "density",
            "dynamic viscosity",
        ],
        "value": [
            air_state.mass_flow,
            air_state.temperature,
            air_state.pressure,
            air_state.props.rho,
            air_state.props.mu * 1e6,
        ],
        "unit": ["kg/s", "K", "Pa", "kg/m³", "µPa·s"],
    }
)

## Flow-section geometry

Circular, rectangular, and custom sections expose area, hydraulic diameter, and area-equivalent circular diameter through the public geometry API. For a rectangle the geometry class supplies `D_h = 2wh / (w + h)`; the notebook does not reimplement that property.

In [ ]:
flow_sections = [
    ("circular: diameter 0.50 m", CircularFlowSection(diameter=0.50)),
    (
        "rectangular: 0.80 m × 0.40 m",
        RectangularFlowSection(width=0.80, height=0.40),
    ),
    (
        "custom: vendor area and D_h",
        CustomFlowSection(area=0.25, hydraulic_diameter=0.45),
    ),
]

pd.DataFrame(
    [
        {
            "section": label,
            "area [m²]": section.flow_area,
            "hydraulic diameter [m]": section.hydraulic_diameter,
            "equivalent circular diameter [m]": section.equivalent_circular_diameter,
        }
        for label, section in flow_sections
    ]
)

## Straight sections

The production straight-section model applies Darcy–Weisbach friction to a constant-area stage. Because one state and one area are used on both sides, `delta_dynamic_pressure = 0` and therefore `dp_static = dp_irreversible`.

In [ ]:
straight_circular_section = CircularFlowSection(diameter=0.50)
straight_result = calculate_straight_section_pressure_drop(
    geometry=StraightSectionGeometry(
        flow_area=straight_circular_section.flow_area,
        hydraulic_diameter=straight_circular_section.hydraulic_diameter,
        length=8.0,  # duct length [m]
        roughness=0.045e-3,  # 0.045 mm [m]
        section_shape=straight_circular_section.section_shape,
    ),
    state=air_state,
    stage_id="straight_circular_duct",
)

assert straight_result.dp_irreversible > 0.0
assert math.isclose(straight_result.delta_dynamic_pressure, 0.0, abs_tol=1e-12)
display(
    pd.DataFrame(
        {
            "quantity": ["flow area [m²]", "hydraulic diameter [m]"],
            "value": [
                straight_circular_section.flow_area,
                straight_circular_section.hydraulic_diameter,
            ],
        }
    )
)
display(show_stage_result(assert_stage_consistent(straight_result)))

In [ ]:
straight_rectangular_section = RectangularFlowSection(width=0.80, height=0.40)
straight_rectangular_result = calculate_straight_section_pressure_drop(
    geometry=StraightSectionGeometry(
        flow_area=straight_rectangular_section.flow_area,
        hydraulic_diameter=straight_rectangular_section.hydraulic_diameter,
        length=8.0,  # duct length [m]
        roughness=0.15e-3,  # 0.15 mm [m]
        section_shape=straight_rectangular_section.section_shape,
    ),
    state=air_state,
    stage_id="straight_rectangular_duct",
)

assert straight_rectangular_result.dp_irreversible > 0.0
assert math.isclose(
    straight_rectangular_result.delta_dynamic_pressure, 0.0, abs_tol=1e-12
)
display(
    pd.DataFrame(
        {
            "quantity": ["flow area [m²]", "hydraulic diameter [m]"],
            "value": [
                straight_rectangular_section.flow_area,
                straight_rectangular_section.hydraulic_diameter,
            ],
        }
    )
)
display(show_stage_result(assert_stage_consistent(straight_rectangular_result)))

## Circular area changes

Expansions decelerate the flow, so `delta_dynamic_pressure < 0`; contractions accelerate it, so `delta_dynamic_pressure > 0`. The irreversible loss remains non-negative in either direction.

A sudden change uses the production correlation's 180° limiting case. A gradual change instead uses an included angle, supplied directly or derived from length, so a gentler transition is treated differently from an abrupt one.

For gradual examples, supplying `length` lets the production model derive its equivalent included angle. The current public geometry and result objects do not expose that derived numeric angle. The cells report this limitation instead of importing the private derivation helper or duplicating its equation.

In [ ]:
expansion_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=CircularFlowSection(
            diameter=0.40,  # upstream circular duct diameter [m]
        ),
        downstream_section=CircularFlowSection(
            diameter=0.80,  # downstream circular duct diameter [m]
        ),
        change_type=AreaChangeType.SUDDEN,
    ),
    state=air_state,
    stage_id="sudden_circular_expansion",
)

assert expansion_result.dp_irreversible > 0.0
assert expansion_result.delta_dynamic_pressure < 0.0
display(show_stage_result(assert_stage_consistent(expansion_result)))

In [ ]:
gradual_expansion_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=CircularFlowSection(diameter=0.40),
        downstream_section=CircularFlowSection(diameter=0.80),
        change_type=AreaChangeType.GRADUAL,
        length=1.50,  # production derives the included angle from this length [m]
    ),
    state=air_state,
    stage_id="gradual_circular_diffuser",
)

display(
    pd.DataFrame(
        [{
            "geometry diagnostic": "included angle",
            "value": "derived internally from length=1.50 m; numeric value is not exposed publicly",
        }]
    )
)
assert gradual_expansion_result.dp_irreversible > 0.0
assert gradual_expansion_result.delta_dynamic_pressure < 0.0
display(show_stage_result(assert_stage_consistent(gradual_expansion_result)))

In [ ]:
contraction_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=CircularFlowSection(
            diameter=0.80,  # upstream large-section diameter [m]
        ),
        downstream_section=CircularFlowSection(
            diameter=0.40,  # downstream small-section diameter [m]
        ),
        change_type=AreaChangeType.SUDDEN,
    ),
    state=air_state,
    stage_id="sudden_circular_contraction",
)

assert contraction_result.dp_irreversible > 0.0
assert contraction_result.delta_dynamic_pressure > 0.0
display(show_stage_result(assert_stage_consistent(contraction_result)))

In [ ]:
gradual_contraction_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=CircularFlowSection(diameter=0.80),
        downstream_section=CircularFlowSection(diameter=0.40),
        change_type=AreaChangeType.GRADUAL,
        length=1.00,  # production derives the included angle from this length [m]
    ),
    state=air_state,
    stage_id="gradual_circular_contraction",
)

display(
    pd.DataFrame(
        [{
            "geometry diagnostic": "included angle",
            "value": "derived internally from length=1.00 m; numeric value is not exposed publicly",
        }]
    )
)
assert gradual_contraction_result.dp_irreversible > 0.0
assert gradual_contraction_result.delta_dynamic_pressure > 0.0
display(show_stage_result(assert_stage_consistent(gradual_contraction_result)))

## Rectangular and mixed-shape transitions

The production model uses exact section areas and velocities. For a non-axisymmetric gradual transition, however, its included angle is based on area-equivalent circular diameters. The returned structured warning identifies this approximation; it is deliberately retained and displayed.

In [ ]:
rectangular_diffuser_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=RectangularFlowSection(width=0.60, height=0.40),
        downstream_section=RectangularFlowSection(width=1.20, height=0.80),
        change_type=AreaChangeType.GRADUAL,
        length=1.20,  # transition length [m]
    ),
    state=air_state,
    stage_id="rectangular_diffuser",
)

assert rectangular_diffuser_result.delta_dynamic_pressure < 0.0
display(show_stage_result(assert_stage_consistent(rectangular_diffuser_result)))

In [ ]:
rectangular_contraction_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=RectangularFlowSection(width=1.20, height=0.80),
        downstream_section=RectangularFlowSection(width=0.60, height=0.40),
        change_type=AreaChangeType.GRADUAL,
        length=1.00,  # transition length [m]
    ),
    state=air_state,
    stage_id="rectangular_contraction",
)

assert rectangular_contraction_result.delta_dynamic_pressure > 0.0
display(show_stage_result(assert_stage_consistent(rectangular_contraction_result)))

In [ ]:
circular_to_rectangular_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=CircularFlowSection(diameter=0.50),
        downstream_section=RectangularFlowSection(width=0.80, height=0.50),
        change_type=AreaChangeType.GRADUAL,
        length=1.00,  # transition length [m]
    ),
    state=air_state,
    stage_id="circular_to_rectangular",
)

assert circular_to_rectangular_result.delta_dynamic_pressure < 0.0
display(show_stage_result(assert_stage_consistent(circular_to_rectangular_result)))

In [ ]:
rectangular_to_circular_result = calculate_area_change_pressure_drop(
    geometry=AreaChangeGeometry(
        upstream_section=RectangularFlowSection(width=0.80, height=0.50),
        downstream_section=CircularFlowSection(diameter=0.50),
        change_type=AreaChangeType.GRADUAL,
        length=1.00,  # transition length [m]
    ),
    state=air_state,
    stage_id="rectangular_to_circular",
)

assert rectangular_to_circular_result.delta_dynamic_pressure > 0.0
display(show_stage_result(assert_stage_consistent(rectangular_to_circular_result)))

## Elbows and bends

The public geometry represents circular/rectangular, smooth-radius/segmented, and any angle through 180°. Automatic geometry-only elbow correlations are not currently implemented. Each supported calculation below therefore uses an explicitly supplied illustrative fitting `K` or `Le/D`; actual design inputs should come from fitting data that matches the construction. A U-bend is the same direction-change model evaluated at 180°, not a separate fundamental model.

`turning_vane_count` records whether a rectangular fitting has vanes, but it does not automatically alter an explicit `K` or `Le/D`. The vaned example therefore supplies its own illustrative fitting-specific input.

In [ ]:
circular_smooth_geometry = CircularElbowGeometry(
    diameter=0.50,
    angle_deg=90.0,
    construction=ElbowConstruction.SMOOTH_RADIUS,
    centerline_radius=1.5 * 0.50,  # R/D = 1.5
    method=DirectionChangeMethod.USER_DEFINED_K,
    loss_coefficient=0.30,  # illustrative fitting-specific K
)
circular_smooth_result = calculate_circular_elbow_pressure_drop(
    geometry=circular_smooth_geometry,
    state=air_state,
    stage_id="circular_smooth_radius_elbow",
)

display(
    pd.DataFrame([{
        "cross-section": "circular",
        "construction": circular_smooth_geometry.construction.value,
        "angle [deg]": circular_smooth_geometry.angle_deg,
        "R/D [-]": circular_smooth_geometry.radius_ratio,
        "input basis": "illustrative user-supplied K",
    }])
)
display(show_stage_result(assert_stage_consistent(circular_smooth_result)))

In [ ]:
circular_segmented_geometry = CircularElbowGeometry(
    diameter=0.50,
    angle_deg=90.0,
    construction=ElbowConstruction.SEGMENTED,
    segment_count=3,
    roughness=0.045e-3,
    method=DirectionChangeMethod.EQUIVALENT_LENGTH,
    equivalent_length_ratio=40.0,  # illustrative fitting Le/D
)
circular_segmented_result = calculate_circular_elbow_pressure_drop(
    geometry=circular_segmented_geometry,
    state=air_state,
    stage_id="circular_segmented_elbow",
)

display(
    pd.DataFrame([{
        "cross-section": "circular",
        "construction": circular_segmented_geometry.construction.value,
        "angle [deg]": circular_segmented_geometry.angle_deg,
        "segments": circular_segmented_geometry.segment_count,
        "illustrative Le/D [-]": circular_segmented_geometry.equivalent_length_ratio,
    }])
)
display(show_stage_result(assert_stage_consistent(circular_segmented_result)))

In [ ]:
rectangular_smooth_geometry = RectangularElbowGeometry(
    width=0.80,
    height=0.40,
    turn_plane=RectangularTurnPlane.WIDTH,
    angle_deg=90.0,
    construction=ElbowConstruction.SMOOTH_RADIUS,
    centerline_radius=1.5 * 0.80,  # radius ratio in the width turn plane
    turning_vane_count=0,
    method=DirectionChangeMethod.USER_DEFINED_K,
    loss_coefficient=0.35,  # illustrative fitting-specific K
)
rectangular_smooth_result = calculate_rectangular_elbow_pressure_drop(
    geometry=rectangular_smooth_geometry,
    state=air_state,
    stage_id="rectangular_smooth_radius_elbow",
)

display(
    pd.DataFrame([{
        "aspect ratio [-]": rectangular_smooth_geometry.aspect_ratio,
        "turn plane": rectangular_smooth_geometry.turn_plane.value,
        "turning dimension [m]": rectangular_smooth_geometry.turning_dimension,
        "radius ratio [-]": rectangular_smooth_geometry.radius_ratio,
        "hydraulic diameter [m]": rectangular_smooth_geometry.hydraulic_diameter,
        "turning vanes": rectangular_smooth_geometry.turning_vane_count,
    }])
)
display(show_stage_result(assert_stage_consistent(rectangular_smooth_result)))

In [ ]:
rectangular_segmented_examples = []
for turning_vanes, illustrative_le_over_dh in [(0, 40.0), (2, 25.0)]:
    geometry = RectangularElbowGeometry(
        width=0.80,
        height=0.40,
        turn_plane=RectangularTurnPlane.WIDTH,
        angle_deg=90.0,
        construction=ElbowConstruction.SEGMENTED,
        segment_count=3,
        turning_vane_count=turning_vanes,
        roughness=0.15e-3,
        method=DirectionChangeMethod.EQUIVALENT_LENGTH,
        equivalent_length_ratio=illustrative_le_over_dh,
    )
    result = calculate_rectangular_elbow_pressure_drop(
        geometry=geometry,
        state=air_state,
        stage_id=f"rectangular_segmented_{turning_vanes}_vanes",
    )
    assert_stage_consistent(result)
    rectangular_segmented_examples.append((geometry, result))

pd.DataFrame(
    [
        {
            "construction": geometry.construction.value,
            "angle [deg]": geometry.angle_deg,
            "segments": geometry.segment_count,
            "turning vanes": geometry.turning_vane_count,
            "aspect ratio [-]": geometry.aspect_ratio,
            "turn plane": geometry.turn_plane.value,
            "turning dimension [m]": geometry.turning_dimension,
            "hydraulic diameter [m]": geometry.hydraulic_diameter,
            "illustrative Le/D_h [-]": geometry.equivalent_length_ratio,
            "method": result.method,
            "K [-]": result.loss_coefficient,
            "dp_irreversible [Pa]": result.dp_irreversible,
            "warnings": "; ".join(w.code for w in result.warnings) or "none",
        }
        for geometry, result in rectangular_segmented_examples
    ]
)

In [ ]:
angle_comparison = []
for angle_deg, illustrative_k in [(45.0, 0.15), (90.0, 0.30), (180.0, 0.60)]:
    geometry = CircularElbowGeometry(
        diameter=0.50,
        angle_deg=angle_deg,
        construction=ElbowConstruction.SMOOTH_RADIUS,
        centerline_radius=1.5 * 0.50,  # same R/D = 1.5
        method=DirectionChangeMethod.USER_DEFINED_K,
        loss_coefficient=illustrative_k,  # illustrative angle-specific fitting data
    )
    result = calculate_circular_elbow_pressure_drop(
        geometry=geometry,
        state=air_state,
        stage_id=f"circular_smooth_{angle_deg:.0f}_deg",
    )
    assert_stage_consistent(result)
    angle_comparison.append(result)

pd.DataFrame(
    [
        {
            "angle [deg]": angle,
            "interpretation": "180° elbow / U-bend" if angle == 180.0 else "elbow",
            "method": result.method,
            "illustrative K [-]": result.loss_coefficient,
            "dp_irreversible [Pa]": result.dp_irreversible,
        }
        for angle, result in zip((45.0, 90.0, 180.0), angle_comparison)
    ]
)

## Flat obstruction from blockage ratio

A 25% blocked bird net is evaluated as a normal, high-Re screen-equivalent obstruction referenced to the gross face velocity. The blockage-only approximation does not encode wire shape, bar thickness, spacing, mesh construction, incidence angle, louver angle, or Reynolds-number effects; the production warning is therefore part of the displayed result.

In [ ]:
obstruction_result = calculate_flat_obstruction_pressure_drop(
    geometry=FlatObstructionGeometry(
        face_section=RectangularFlowSection(width=1.0, height=0.6),
        blockage_ratio=0.25,  # 25% of gross face area blocked
        obstruction_type=FlatObstructionType.BIRD_NET,
    ),
    state=air_state,
    stage_id="bird_net_25_percent_blockage",
)

assert obstruction_result.dp_irreversible >= 0.0
assert math.isclose(obstruction_result.delta_dynamic_pressure, 0.0, abs_tol=1e-12)
display(
    pd.DataFrame(
        {
            "quantity": ["gross area [m²]", "face velocity [m/s]"],
            "value": [
                obstruction_result.reference_area,
                obstruction_result.reference_velocity,
            ],
        }
    )
)
display(show_stage_result(assert_stage_consistent(obstruction_result)))

In [ ]:
blockage_comparison = []
for blockage_ratio in (0.00, 0.10, 0.25, 0.50, 0.75):
    result = calculate_flat_obstruction_pressure_drop(
        geometry=FlatObstructionGeometry(
            face_section=RectangularFlowSection(width=1.0, height=0.6),
            blockage_ratio=blockage_ratio,
            obstruction_type=FlatObstructionType.BIRD_NET,
        ),
        state=air_state,
        stage_id=f"bird_net_{100 * blockage_ratio:.0f}_percent",
    )
    assert_stage_consistent(result)
    blockage_comparison.append(result)

pd.DataFrame(
    [
        {
            "blockage [%]": 100.0 * result.blockage_ratio,
            "open area [%]": 100.0 * result.open_area_ratio,
            "K [-]": result.loss_coefficient,
            "dp_irreversible [Pa]": result.dp_irreversible,
        }
        for result in blockage_comparison
    ]
)

## User-defined losses

For a user-defined `K`, the caller must ensure that both the coefficient definition and reference velocity match the source data. A fixed irreversible pressure drop is useful when a manufacturer supplies a value directly at the selected operating point, but it does not automatically scale to another flow rate.

In [ ]:
user_k_result = calculate_user_defined_pressure_drop(
    geometry=UserDefinedPressureDropGeometry(
        loss_coefficient=1.2,  # user-supplied K
        reference_area=0.30,  # matching K reference area [m²]
        description="illustrative fitting-specific K",
    ),
    state=air_state,
    stage_id="user_defined_k",
)

display(show_stage_result(assert_stage_consistent(user_k_result)))

In [ ]:
fixed_dp_result = calculate_user_defined_pressure_drop(
    geometry=UserDefinedPressureDropGeometry(
        pressure_drop=250.0,  # fixed irreversible loss [Pa]
        description="manufacturer value at this operating point",
    ),
    state=air_state,
    stage_id="user_defined_fixed_pressure_drop",
)

assert math.isclose(fixed_dp_result.dp_irreversible, 250.0)
display(show_stage_result(assert_stage_consistent(fixed_dp_result)))

## Mixed ordered assembly

An assembly preserves the supplied stage order. The table keeps irreversible loss, signed dynamic-pressure change, and signed static-pressure difference separate; `assembly_result.dp_irreversible` is the summed hydraulic resistance.

In [ ]:
assembly_inlet_section = RectangularFlowSection(width=0.80, height=0.40)
assembly_result = calculate_pressure_drop_assembly(
    group_id="mixed_local_assembly",
    geometry=PressureDropAssemblyGeometry(
        stages=(
            StraightSectionGeometry(
                flow_area=assembly_inlet_section.flow_area,
                hydraulic_diameter=assembly_inlet_section.hydraulic_diameter,
                length=3.0,  # rectangular duct length [m]
                roughness=0.15e-3,
                section_shape=assembly_inlet_section.section_shape,
            ),
            AreaChangeGeometry(
                upstream_section=RectangularFlowSection(width=0.80, height=0.40),
                downstream_section=RectangularFlowSection(width=0.60, height=0.30),
                change_type=AreaChangeType.GRADUAL,
                length=0.80,  # contraction length [m]
            ),
            RectangularElbowGeometry(
                width=0.60,
                height=0.30,
                turn_plane=RectangularTurnPlane.WIDTH,
                angle_deg=90.0,
                construction=ElbowConstruction.SMOOTH_RADIUS,
                centerline_radius=1.5 * 0.60,
                turning_vane_count=0,
                method=DirectionChangeMethod.USER_DEFINED_K,
                loss_coefficient=0.40,  # illustrative fitting-specific K
            ),
            FlatObstructionGeometry(
                face_section=RectangularFlowSection(width=0.60, height=0.30),
                blockage_ratio=0.15,
                obstruction_type=FlatObstructionType.BIRD_NET,
            ),
            UserDefinedPressureDropGeometry(
                loss_coefficient=0.60,
                reference_area=0.18,  # downstream duct area [m²]
                description="additional fitting",
            ),
        )
    ),
    state=air_state,
)

In [ ]:
for stage in assembly_result.stages:
    assert_stage_consistent(stage)

assert math.isclose(
    assembly_result.dp_irreversible,
    sum(stage.dp_irreversible for stage in assembly_result.stages),
    rel_tol=1e-12,
)

display(
    pd.DataFrame(
        [
            {
                "stage ID": stage.stage_id,
                "stage type": stage.stage_type,
                "method": stage.method,
                "dp_irreversible [Pa]": stage.dp_irreversible,
                "delta_dynamic_pressure [Pa]": stage.delta_dynamic_pressure,
                "dp_static [Pa]": stage.dp_static,
                "warnings": "; ".join(
                    f"{warning.code}: {warning.message}" for warning in stage.warnings
                ) or "none",
            }
            for stage in assembly_result.stages
        ]
    )
)
display(
    pd.DataFrame(
        [{
            "group": assembly_result.group_id,
            "dp_irreversible [Pa]": assembly_result.dp_irreversible,
            "delta_dynamic_pressure [Pa]": assembly_result.delta_dynamic_pressure,
            "dp_static [Pa]": assembly_result.dp_static,
        }]
    )
)

## Complete explicit tube-side path

The advanced example first obtains a real one-pass core result from public `calculate_tube_bundle_hydraulics`, then combines explicit inlet and outlet assemblies with it. With one tube pass, both `returns` and `return_states` are empty.

The local area changes below represent illustrative external header/nozzle transitions into and out of the aggregate core flow area. They do not replace the tube-sheet entrance and exit losses already included by the tube-bundle core calculation. This remains an explicit application-layer call; no exchanger `solve()`, `simulate()`, or `rate()` operation evaluates the path automatically.

In [ ]:
tube_bundle_core = calculate_tube_bundle_hydraulics(
    m_dot=5.0,
    flow_area_per_pass=0.10,  # total parallel tube area per pass [m²]
    hydraulic_diameter=0.02,  # tube inside diameter [m]
    hydraulic_length_total=8.0,  # one-pass flow length [m]
    n_tube_passes=1,
    roughness_inner=0.045e-3,
    inlet_props=air_state.props,
)
tube_core_section = CustomFlowSection(
    area=tube_bundle_core.flow_area_per_pass,
    hydraulic_diameter=tube_bundle_core.hydraulic_diameter,
)

explicit_path_result = calculate_tube_side_pressure_drop_path(
    tube_bundle=tube_bundle_core,
    n_tube_passes=1,
    path=SpecifiedTubeSidePressureDropPath(
        inlet=PressureDropAssemblyGeometry(
            stages=(
                AreaChangeGeometry(
                    upstream_section=CircularFlowSection(diameter=0.50),
                    downstream_section=tube_core_section,
                    change_type=AreaChangeType.SUDDEN,
                ),
            )
        ),
        returns=(),
        outlet=PressureDropAssemblyGeometry(
            stages=(
                AreaChangeGeometry(
                    upstream_section=tube_core_section,
                    downstream_section=CircularFlowSection(diameter=0.50),
                    change_type=AreaChangeType.SUDDEN,
                ),
            )
        ),
    ),
    inlet_state=air_state,
    return_states=(),
    outlet_state=air_state,
)

In [ ]:
for group in explicit_path_result.groups:
    for stage in group.stages:
        assert_stage_consistent(stage)

assert math.isclose(
    explicit_path_result.dp_total,
    explicit_path_result.dp_core + explicit_path_result.dp_local,
    rel_tol=1e-12,
)
assert math.isclose(
    explicit_path_result.dp_local,
    sum(
        group.dp_irreversible
        for group in explicit_path_result.groups
        if group.group_id != "tube_bundle"
    ),
    rel_tol=1e-12,
)
assert math.isclose(
    explicit_path_result.dp_static_total,
    explicit_path_result.dp_total
    + explicit_path_result.delta_dynamic_pressure_total,
    rel_tol=1e-12,
)

display(
    pd.DataFrame(
        [
            {
                "group": group.group_id,
                "stage type": stage.stage_type,
                "method": stage.method,
                "dp_irreversible [Pa]": stage.dp_irreversible,
                "delta_dynamic_pressure [Pa]": stage.delta_dynamic_pressure,
                "dp_static [Pa]": stage.dp_static,
                "warnings": "; ".join(w.code for w in stage.warnings) or "none",
            }
            for group in explicit_path_result.groups
            for stage in group.stages
        ]
    )
)
display(
    pd.DataFrame(
        {
            "scope": ["core", "local", "complete path"],
            "dp_irreversible [Pa]": [
                explicit_path_result.dp_core,
                explicit_path_result.dp_local,
                explicit_path_result.dp_total,
            ],
            "delta_dynamic_pressure [Pa]": [
                explicit_path_result.delta_dynamic_pressure_core,
                explicit_path_result.delta_dynamic_pressure_local,
                explicit_path_result.delta_dynamic_pressure_total,
            ],
            "dp_static [Pa]": [
                explicit_path_result.dp_static_core,
                explicit_path_result.dp_static_local,
                explicit_path_result.dp_static_total,
            ],
        }
    )
)

## Interpretation and current limits

- Hydraulic resistance is aggregated from `dp_irreversible`, never by adding `delta_dynamic_pressure` again.
- The public API currently does not expose the numeric included angle derived from transition length; this notebook leaves derivation inside production code and reports that diagnostic limitation explicitly.
- Automatic elbow geometry correlations are not implemented, so the illustrated `K` and `Le/D` inputs must be replaced with fitting-specific data for design work.
- The flat-obstruction result retains its structured high-Re blockage-only limitation warning.
- Complete local paths are explicit application-layer calculations and remain separate from the standard heat-exchanger solver workflow.